In [ ]:
#!pip install -q kagglehub

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [ ]:
import kagglehub

path = kagglehub.dataset_download('aliafzal9323/chicago-crime-dataset-2024-2026')
print('Path to dataset files:', path)

csv_files = glob.glob(os.path.join(path, '*.csv'))
print('Found:', csv_files)
DATA_PATH = csv_files[0]

df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    dtype={'IUCR': str},
    parse_dates=['Date'],
)
print(f'{df.shape[0]:,} rows x {df.shape[1]} columns')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())
df.head(3)

Using Colab cache for faster access to the 'chicago-crime-dataset-2024-2026' dataset.
Path to dataset files: /kaggle/input/chicago-crime-dataset-2024-2026
Found: ['/kaggle/input/chicago-crime-dataset-2024-2026/chicago crimes.csv']


In [ ]:
#missing values
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': df.isnull().mean().mul(100).round(2)
}).query('missing_count > 0').sort_values('missing_count', ascending=False)

display(missing)

In [ ]:
#duplicates and target distributions
print(f'duplicate rows: {df.duplicated().sum():,}')

if 'case_number' in df.columns:
    print(f'duplicate case numbers: {df["case_number"].duplicated().sum():,}')

print('\narrest:')
print(df['arrest'].value_counts(normalize=True).mul(100).round(2))

print('\nprimary_type (top 20):')
print(df['primary_type'].value_counts().head(20))

In [ ]:
#cleaning
df_clean = df.copy()

# drop exact duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'removed {before - len(df_clean):,} duplicate rows')

# drop rows with unparseable dates
if not pd.api.types.is_datetime64_any_dtype(df_clean['date']):
    df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')
before = len(df_clean)
df_clean = df_clean.dropna(subset=['date'])
print(f'removed {before - len(df_clean):,} rows with bad dates')

# fix IUCR
df_clean['iucr'] = df_clean['iucr'].str.strip().str.zfill(4)

# location_description: flag and fill missing
df_clean['location_unknown_flag'] = df_clean['location_description'].isna().astype(int)
df_clean['location_description'] = df_clean['location_description'].fillna('UNKNOWN')

# coordinates: flag missing but keep rows
df_clean['coords_missing_flag'] = df_clean['latitude'].isna().astype(int)

# district: only ~47 missing, just drop
before = len(df_clean)
df_clean = df_clean.dropna(subset=['district'])
print(f'removed {before - len(df_clean):,} rows with missing district')

# ward and community_area: ~614k missing, fill with -1
for col in ['ward', 'community_area']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(-1)

# group rare primary_type categories into OTHER
type_freq = df_clean['primary_type'].value_counts(normalize=True)
rare_types = type_freq[type_freq < 0.005].index
df_clean['primary_type_grouped'] = df_clean['primary_type'].where(
    ~df_clean['primary_type'].isin(rare_types), 'OTHER'
)
print(f'primary_type: {df_clean["primary_type"].nunique()} -> {df_clean["primary_type_grouped"].nunique()} after grouping rare')
print(f'final shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

In [ ]:
# filter to 2024-2026 to match project scope
# full dataset goes back to 2001 but proposal scopes to recent years
df_clean = df_clean[df_clean['date'].dt.year.between(2024, 2026)]
print(f'filtered shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#Saved clean data

In [ ]:
df_clean.to_parquet('/content/chicago_crimes_clean.parquet', index=False)
print('saved')

Note: cells above only need to be run once to generate the parquet.
If reloading in a new session, run the reload cell below and skip to visualizations.


#missing value heatmap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#missing value heatmap
sample = df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(sample.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=ax)
ax.set_title('Missing Value Pattern (5,000-row sample)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('viz1_missing.png', dpi=150)
plt.show()

#arrest rate by crime type

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#arrest rate by crime type
arrest_by_type = (
    df_clean.groupby('primary_type_grouped')['arrest']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'arrest_rate', 'count': 'n_incidents'})
    .query('n_incidents >= 1000')
    .sort_values('arrest_rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(
    arrest_by_type.index,
    arrest_by_type['arrest_rate'] * 100,
    color=sns.color_palette('coolwarm', len(arrest_by_type))
)
ax.axvline(x=df_clean['arrest'].mean() * 100, color='black', linestyle='--', label='overall average')
ax.set_xlabel('Arrest Rate (%)')
ax.set_title('Arrest Rate by Crime Type')
ax.legend()
plt.tight_layout()
plt.savefig('viz2_arrest_by_type.png', dpi=150)
plt.show()

#crime volume and arrest rate by hour

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#crime volume and arrest rate by hour
df_clean['hour_of_day'] = df_clean['date'].dt.hour

hourly_counts = df_clean.groupby(['hour_of_day', 'arrest']).size().unstack(fill_value=0)
hourly_counts.columns = ['No Arrest', 'Arrest']
arrest_rate_hour = df_clean.groupby('hour_of_day')['arrest'].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

hourly_counts.plot(kind='bar', ax=ax1, stacked=True, colormap='Set2', width=0.8)
ax1.set_title('Crime Volume by Hour of Day')
ax1.set_xlabel('Hour (0 = midnight)')
ax1.set_ylabel('Incident Count')
ax1.set_xticklabels(range(24), rotation=0, fontsize=8)

ax2.plot(arrest_rate_hour.index, arrest_rate_hour.values, marker='o', color='steelblue', linewidth=2)
ax2.set_title('Arrest Rate (%) by Hour of Day')
ax2.set_xlabel('Hour (0 = midnight)')
ax2.set_ylabel('Arrest Rate (%)')
ax2.set_xticks(range(24))

plt.tight_layout()
plt.savefig('viz3_hourly.png', dpi=150)
plt.show()

#class distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#class distribution
type_counts = df_clean['primary_type_grouped'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(type_counts)), type_counts.values,
       color=sns.color_palette('tab20', len(type_counts)))
ax.set_xticks(range(len(type_counts)))
ax.set_xticklabels(type_counts.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Incidents')
ax.set_title('Class Distribution - Primary Crime Type (Grouped)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('viz4_class_dist.png', dpi=150)
plt.show()

#geographic density


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#geographic density
spatial = df_clean.dropna(subset=['latitude', 'longitude'])
spatial = spatial[
    spatial['latitude'].between(41.6, 42.1) &
    spatial['longitude'].between(-87.95, -87.5)
]
sample_spatial = spatial.sample(min(100_000, len(spatial)), random_state=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

ax1.scatter(sample_spatial['longitude'], sample_spatial['latitude'],
            alpha=0.05, s=0.5, c='steelblue')
ax1.set_title('All Incidents (100k sample)')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')

for label, color in [(True, 'crimson'), (False, 'steelblue')]:
    sub = sample_spatial[sample_spatial['arrest'] == label]
    ax2.scatter(sub['longitude'], sub['latitude'],
                alpha=0.05, s=0.5, c=color, label=f'arrest={label}')
ax2.set_title('Arrest vs. No Arrest')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.legend(markerscale=10, framealpha=0.8)

plt.tight_layout()
plt.savefig('viz5_geo.png', dpi=150)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

# zoomed crime map - downtown chicago / loop area
loop = df_clean.dropna(subset=['latitude', 'longitude'])
loop = loop[
    loop['latitude'].between(41.83, 41.90) &
    loop['longitude'].between(-87.65, -87.60)
]
print(f'incidents in loop area: {len(loop):,}')

sample_loop = loop.sample(min(50_000, len(loop)), random_state=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# all incidents
ax1.scatter(sample_loop['longitude'], sample_loop['latitude'],
            alpha=0.1, s=1, c='steelblue')
ax1.set_title('All Incidents - Loop Area')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')

# arrest vs no arrest
for label, color in [(True, 'crimson'), (False, 'steelblue')]:
    sub = sample_loop[sample_loop['arrest'] == label]
    ax2.scatter(sub['longitude'], sub['latitude'],
                alpha=0.1, s=1, c=color, label=f'arrest={label}')
ax2.set_title('Arrest vs. No Arrest - Loop Area')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.legend(markerscale=10, framealpha=0.8)

plt.tight_layout()
plt.savefig('viz8_loop_geo.png', dpi=150)
plt.show()

#arrest rate by day of week

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#arrest rate by day of week
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

arrest_by_day = df_clean.groupby('day_of_week')['arrest'].mean() * 100
volume_by_day = df_clean.groupby('day_of_week').size()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(day_labels, volume_by_day.values, color=sns.color_palette('pastel', 7))
ax1.set_title('Crime Volume by Day of Week')
ax1.set_ylabel('Incident Count')

ax2.plot(day_labels, arrest_by_day.values, marker='s', color='darkorange', linewidth=2.5)
ax2.set_title('Arrest Rate (%) by Day of Week')
ax2.set_ylabel('Arrest Rate (%)')

plt.tight_layout()
plt.savefig('viz6_dow.png', dpi=150)
plt.show()

#crime volume and arrest rate by month

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')


#crime volume and arrest rate by month
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
print(f'reloaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

df_clean['month'] = df_clean['date'].dt.month
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

volume_by_month = df_clean.groupby('month').size()
arrest_by_month = df_clean.groupby('month')['arrest'].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(month_labels, volume_by_month.values, color=sns.color_palette('pastel', 12))
ax1.set_title('Crime Volume by Month')
ax1.set_ylabel('Incident Count')

ax2.plot(month_labels, arrest_by_month.values, marker='s', color='steelblue', linewidth=2.5)
ax2.set_title('Arrest Rate (%) by Month')
ax2.set_ylabel('Arrest Rate (%)')

plt.tight_layout()
plt.savefig('viz7_monthly.png', dpi=150)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')


df_year_filtered = df_clean[df_clean['year'].between(2024, 2026)]

monthly_by_year = df_year_filtered.groupby(['year', 'month']).size().unstack(level=0)
monthly_by_year.index = month_labels[:len(monthly_by_year)]

fig, ax = plt.subplots(figsize=(12, 5))
monthly_by_year.plot(kind='bar', ax=ax, width=0.7)
ax.set_title('Crime Volume by Month, Split by Year (2024-2026)')
ax.set_ylabel('Incident Count')
ax.set_xticklabels(monthly_by_year.index, rotation=0)
ax.legend(title='Year')
plt.tight_layout()
plt.savefig('viz7b_monthly_by_year.png', dpi=150)
plt.show()

#neighborhood

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')


# community area number to name mapping
community_area_names = {
    1: 'Rogers Park', 2: 'West Ridge', 3: 'Uptown', 4: 'Lincoln Square',
    5: 'North Center', 6: 'Lake View', 7: 'Lincoln Park', 8: 'Near North Side',
    9: 'Edison Park', 10: 'Norwood Park', 11: 'Jefferson Park', 12: 'Forest Glen',
    13: 'North Park', 14: 'Albany Park', 15: 'Portage Park', 16: 'Irving Park',
    17: 'Dunning', 18: 'Montclare', 19: 'Belmont Cragin', 20: 'Hermosa',
    21: 'Avondale', 22: 'Logan Square', 23: 'Humboldt Park', 24: 'West Town',
    25: 'Austin', 26: 'West Garfield Park', 27: 'East Garfield Park', 28: 'Near West Side',
    29: 'North Lawndale', 30: 'South Lawndale', 31: 'Lower West Side', 32: 'Loop',
    33: 'Near South Side', 34: 'Armour Square', 35: 'Douglas', 36: 'Oakland',
    37: 'Fuller Park', 38: 'Grand Boulevard', 39: 'Kenwood', 40: 'Washington Park',
    41: 'Hyde Park', 42: 'Woodlawn', 43: 'South Shore', 44: 'Chatham',
    45: 'Avalon Park', 46: 'South Chicago', 47: 'Burnside', 48: 'Calumet Heights',
    49: 'Roseland', 50: 'Pullman', 51: 'South Deering', 52: 'East Side',
    53: 'West Pullman', 54: 'Riverdale', 55: 'Hegewisch', 56: 'Garfield Ridge',
    57: 'Archer Heights', 58: 'Brighton Park', 59: 'McKinley Park', 60: 'Bridgeport',
    61: 'New City', 62: 'West Elsdon', 63: 'Gage Park', 64: 'Clearing',
    65: 'West Lawn', 66: 'Chicago Lawn', 67: 'West Englewood', 68: 'Englewood',
    69: 'Greater Grand Crossing', 70: 'Ashburn', 71: 'Auburn Gresham',
    72: 'Beverly', 73: 'Washington Heights', 74: 'Mount Greenwood',
    75: 'Morgan Park', 76: 'OHare', 77: 'Edgewater'
}

df_clean['neighborhood'] = df_clean['community_area'].map(community_area_names)

# top 15 neighborhoods by crime volume
top_neighborhoods = (
    df_clean[df_clean['community_area'] != -1]
    .groupby('neighborhood')
    .agg(
        total_crimes=('arrest', 'count'),
        arrest_rate=('arrest', 'mean')
    )
    .sort_values('total_crimes', ascending=False)
    .head(15)
)
top_neighborhoods['arrest_rate'] = (top_neighborhoods['arrest_rate'] * 100).round(1)
print(top_neighborhoods)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')



fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# crime volume
top_neighborhoods['total_crimes'].plot(kind='barh', ax=ax1, color='steelblue')
ax1.invert_yaxis()
ax1.set_title('Top 15 Neighborhoods by Crime Volume')
ax1.set_xlabel('Total Incidents')

# arrest rate for same neighborhoods
top_neighborhoods['arrest_rate'].plot(kind='barh', ax=ax2, color='darkorange')
ax2.invert_yaxis()
ax2.set_title('Arrest Rate (%) in Those Neighborhoods')
ax2.set_xlabel('Arrest Rate (%)')

plt.tight_layout()
plt.savefig('viz_neighborhoods.png', dpi=150)
plt.show()

In [ ]:
# community area number to name mapping
community_area_names = {
    1: 'Rogers Park', 2: 'West Ridge', 3: 'Uptown', 4: 'Lincoln Square',
    5: 'North Center', 6: 'Lake View', 7: 'Lincoln Park', 8: 'Near North Side',
    9: 'Edison Park', 10: 'Norwood Park', 11: 'Jefferson Park', 12: 'Forest Glen',
    13: 'North Park', 14: 'Albany Park', 15: 'Portage Park', 16: 'Irving Park',
    17: 'Dunning', 18: 'Montclare', 19: 'Belmont Cragin', 20: 'Hermosa',
    21: 'Avondale', 22: 'Logan Square', 23: 'Humboldt Park', 24: 'West Town',
    25: 'Austin', 26: 'West Garfield Park', 27: 'East Garfield Park', 28: 'Near West Side',
    29: 'North Lawndale', 30: 'South Lawndale', 31: 'Lower West Side', 32: 'Loop',
    33: 'Near South Side', 34: 'Armour Square', 35: 'Douglas', 36: 'Oakland',
    37: 'Fuller Park', 38: 'Grand Boulevard', 39: 'Kenwood', 40: 'Washington Park',
    41: 'Hyde Park', 42: 'Woodlawn', 43: 'South Shore', 44: 'Chatham',
    45: 'Avalon Park', 46: 'South Chicago', 47: 'Burnside', 48: 'Calumet Heights',
    49: 'Roseland', 50: 'Pullman', 51: 'South Deering', 52: 'East Side',
    53: 'West Pullman', 54: 'Riverdale', 55: 'Hegewisch', 56: 'Garfield Ridge',
    57: 'Archer Heights', 58: 'Brighton Park', 59: 'McKinley Park', 60: 'Bridgeport',
    61: 'New City', 62: 'West Elsdon', 63: 'Gage Park', 64: 'Clearing',
    65: 'West Lawn', 66: 'Chicago Lawn', 67: 'West Englewood', 68: 'Englewood',
    69: 'Greater Grand Crossing', 70: 'Ashburn', 71: 'Auburn Gresham',
    72: 'Beverly', 73: 'Washington Heights', 74: 'Mount Greenwood',
    75: 'Morgan Park', 76: 'OHare', 77: 'Edgewater'
}

df_clean['neighborhood'] = df_clean['community_area'].map(community_area_names)



print(df_clean[df_clean['community_area'] == 42]['primary_type'].value_counts().head(10))
print(f"\ntotal incidents: {len(df_clean[df_clean['community_area'] == 42]):,}")
print(f"arrest rate: {df_clean[df_clean['community_area'] == 42]['arrest'].mean()*100:.1f}%")

neighborhood_stats = (
    df_clean[df_clean['community_area'] != -1]
    .groupby('neighborhood')
    .agg(total=('arrest', 'count'), arrest_rate=('arrest', 'mean'))
    .sort_values('total', ascending=False)
)
neighborhood_stats['arrest_rate'] = (neighborhood_stats['arrest_rate'] * 100).round(1)
print(neighborhood_stats.head(40))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# add neighborhood names
community_area_names = {
    1: 'Rogers Park', 2: 'West Ridge', 3: 'Uptown', 4: 'Lincoln Square',
    5: 'North Center', 6: 'Lake View', 7: 'Lincoln Park', 8: 'Near North Side',
    9: 'Edison Park', 10: 'Norwood Park', 11: 'Jefferson Park', 12: 'Forest Glen',
    13: 'North Park', 14: 'Albany Park', 15: 'Portage Park', 16: 'Irving Park',
    17: 'Dunning', 18: 'Montclare', 19: 'Belmont Cragin', 20: 'Hermosa',
    21: 'Avondale', 22: 'Logan Square', 23: 'Humboldt Park', 24: 'West Town',
    25: 'Austin', 26: 'West Garfield Park', 27: 'East Garfield Park', 28: 'Near West Side',
    29: 'North Lawndale', 30: 'South Lawndale', 31: 'Lower West Side', 32: 'Loop',
    33: 'Near South Side', 34: 'Armour Square', 35: 'Douglas', 36: 'Oakland',
    37: 'Fuller Park', 38: 'Grand Boulevard', 39: 'Kenwood', 40: 'Washington Park',
    41: 'Hyde Park', 42: 'Woodlawn', 43: 'South Shore', 44: 'Chatham',
    45: 'Avalon Park', 46: 'South Chicago', 47: 'Burnside', 48: 'Calumet Heights',
    49: 'Roseland', 50: 'Pullman', 51: 'South Deering', 52: 'East Side',
    53: 'West Pullman', 54: 'Riverdale', 55: 'Hegewisch', 56: 'Garfield Ridge',
    57: 'Archer Heights', 58: 'Brighton Park', 59: 'McKinley Park', 60: 'Bridgeport',
    61: 'New City', 62: 'West Elsdon', 63: 'Gage Park', 64: 'Clearing',
    65: 'West Lawn', 66: 'Chicago Lawn', 67: 'West Englewood', 68: 'Englewood',
    69: 'Greater Grand Crossing', 70: 'Ashburn', 71: 'Auburn Gresham',
    72: 'Beverly', 73: 'Washington Heights', 74: 'Mount Greenwood',
    75: 'Morgan Park', 76: 'OHare', 77: 'Edgewater'
}
df_clean['neighborhood'] = df_clean['community_area'].map(community_area_names)

print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')


# crime type by neighborhood heatmap
# filter out unknown community areas and get top 15 neighborhoods by volume
top_15 = (
    df_clean[df_clean['community_area'] != -1]
    .groupby('neighborhood')
    .size()
    .sort_values(ascending=False)
    .head(15)
    .index
)

# get top 10 crime types for readability
top_10_crimes = df_clean['primary_type'].value_counts().head(10).index

heatmap_data = (
    df_clean[
        df_clean['neighborhood'].isin(top_15) &
        df_clean['primary_type'].isin(top_10_crimes)
    ]
    .groupby(['neighborhood', 'primary_type'])
    .size()
    .unstack(fill_value=0)
)

# normalize by row so each neighborhood shows crime mix not just volume
heatmap_normalized = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_normalized,
    cmap='YlOrRd',
    annot=True,
    fmt='.1f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': '% of neighborhood crimes'}
)
ax.set_title('Crime Type Mix by Neighborhood (% of total incidents)')
ax.set_xlabel('Crime Type')
ax.set_ylabel('Neighborhood')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('viz_neighborhood_heatmap.png', dpi=150)
plt.show()

#violent crime by neighborhood

In [ ]:
 # crime severity group (4 tiers)
VIOLENT_SEVERE = {
    'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'KIDNAPPING', 'HUMAN TRAFFICKING'
}
VIOLENT_MODERATE = {
    'ROBBERY', 'BATTERY', 'ASSAULT', 'STALKING'
}
PROPERTY = {
    'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON',
    'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'
}

def assign_severity(crime_type):
    if crime_type in VIOLENT_SEVERE:
        return 'VIOLENT_SEVERE'
    elif crime_type in VIOLENT_MODERATE:
        return 'VIOLENT_MODERATE'
    elif crime_type in PROPERTY:
        return 'PROPERTY'
    else:
        return 'OTHER'

df_clean['crime_severity_group'] = df_clean['primary_type'].map(assign_severity)



violent_by_neighborhood = (
    df_clean[
        (df_clean['community_area'] != -1) &
        (df_clean['crime_severity_group'].isin(['VIOLENT_SEVERE', 'VIOLENT_MODERATE']))
    ]
    .groupby('neighborhood')
    .agg(
        violent_crimes=('crime_severity_group', 'count'),
        severe_crimes=('crime_severity_group', lambda x: (x == 'VIOLENT_SEVERE').sum()),
        moderate_crimes=('crime_severity_group', lambda x: (x == 'VIOLENT_MODERATE').sum()),
        arrest_rate=('arrest', 'mean')
    )
    .sort_values('violent_crimes', ascending=False)
    .head(15)
)
violent_by_neighborhood['arrest_rate'] = (violent_by_neighborhood['arrest_rate'] * 100).round(1)
print(violent_by_neighborhood)


fig, ax = plt.subplots(figsize=(12, 7))

violent_by_neighborhood[['severe_crimes', 'moderate_crimes']].plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=['crimson', 'darkorange'],
    width=0.7
)
ax.invert_yaxis()
ax.set_title('Top 15 Neighborhoods by Violent Crime (2024-2026)')
ax.set_xlabel('Number of Incidents')
ax.legend(['Violent Severe', 'Violent Moderate'])
plt.tight_layout()
plt.savefig('viz_violent_neighborhoods.png', dpi=150)
plt.show()

#Feature Engineering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')

# recreate time columns
df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year
df_clean['hour_of_day'] = df_clean['date'].dt.hour
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)
df_clean['is_rush_hour'] = (
    (df_clean['is_weekend'] == 0) &
    (df_clean['hour_of_day'].between(7, 9) | df_clean['hour_of_day'].between(16, 19))
).astype(int)

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# neighborhood mapping
community_area_names = {
    1: 'Rogers Park', 2: 'West Ridge', 3: 'Uptown', 4: 'Lincoln Square',
    5: 'North Center', 6: 'Lake View', 7: 'Lincoln Park', 8: 'Near North Side',
    9: 'Edison Park', 10: 'Norwood Park', 11: 'Jefferson Park', 12: 'Forest Glen',
    13: 'North Park', 14: 'Albany Park', 15: 'Portage Park', 16: 'Irving Park',
    17: 'Dunning', 18: 'Montclare', 19: 'Belmont Cragin', 20: 'Hermosa',
    21: 'Avondale', 22: 'Logan Square', 23: 'Humboldt Park', 24: 'West Town',
    25: 'Austin', 26: 'West Garfield Park', 27: 'East Garfield Park', 28: 'Near West Side',
    29: 'North Lawndale', 30: 'South Lawndale', 31: 'Lower West Side', 32: 'Loop',
    33: 'Near South Side', 34: 'Armour Square', 35: 'Douglas', 36: 'Oakland',
    37: 'Fuller Park', 38: 'Grand Boulevard', 39: 'Kenwood', 40: 'Washington Park',
    41: 'Hyde Park', 42: 'Woodlawn', 43: 'South Shore', 44: 'Chatham',
    45: 'Avalon Park', 46: 'South Chicago', 47: 'Burnside', 48: 'Calumet Heights',
    49: 'Roseland', 50: 'Pullman', 51: 'South Deering', 52: 'East Side',
    53: 'West Pullman', 54: 'Riverdale', 55: 'Hegewisch', 56: 'Garfield Ridge',
    57: 'Archer Heights', 58: 'Brighton Park', 59: 'McKinley Park', 60: 'Bridgeport',
    61: 'New City', 62: 'West Elsdon', 63: 'Gage Park', 64: 'Clearing',
    65: 'West Lawn', 66: 'Chicago Lawn', 67: 'West Englewood', 68: 'Englewood',
    69: 'Greater Grand Crossing', 70: 'Ashburn', 71: 'Auburn Gresham',
    72: 'Beverly', 73: 'Washington Heights', 74: 'Mount Greenwood',
    75: 'Morgan Park', 76: 'OHare', 77: 'Edgewater'
}
df_clean['neighborhood'] = df_clean['community_area'].map(community_area_names)

# crime severity group (4 tiers)
VIOLENT_SEVERE = {
    'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'KIDNAPPING', 'HUMAN TRAFFICKING'
}
VIOLENT_MODERATE = {
    'ROBBERY', 'BATTERY', 'ASSAULT', 'STALKING'
}
PROPERTY = {
    'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON',
    'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'
}

def assign_severity(crime_type):
    if crime_type in VIOLENT_SEVERE:
        return 'VIOLENT_SEVERE'
    elif crime_type in VIOLENT_MODERATE:
        return 'VIOLENT_MODERATE'
    elif crime_type in PROPERTY:
        return 'PROPERTY'
    else:
        return 'OTHER'

df_clean['crime_severity_group'] = df_clean['primary_type'].map(assign_severity)

# drop before rejoining to avoid duplicate column error if already in parquet
if 'location_risk_score' in df_clean.columns:
    df_clean = df_clean.drop(columns=['location_risk_score'])
if 'district_arrest_rate' in df_clean.columns:
    df_clean = df_clean.drop(columns=['district_arrest_rate'])

# location_risk_score: arrest rate per location type
# recompute inside training folds only in final pipeline to avoid leakage
loc_arrest_rate = (
    df_clean.groupby('location_description')['arrest']
    .mean()
    .rename('location_risk_score')
)
df_clean = df_clean.join(loc_arrest_rate, on='location_description')

# district_arrest_rate
district_arrest_rate = (
    df_clean.groupby('district')['arrest']
    .mean()
    .rename('district_arrest_rate')
)
df_clean = df_clean.join(district_arrest_rate, on='district')

engineered = [
    'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'is_rush_hour',
    'crime_severity_group', 'location_risk_score', 'district_arrest_rate',
    'location_unknown_flag', 'coords_missing_flag', 'neighborhood'
]
for f in engineered:
    print(f'{f}: {df_clean[f].dtype}, {df_clean[f].nunique()} unique values')

In [ ]:
#save
import os

SAVE_PATH = '/content/chicago_crimes_clean.parquet'
df_clean.to_parquet(SAVE_PATH, index=False)
print(f'saved to {SAVE_PATH}')
print(f'final shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

check = pd.read_parquet(SAVE_PATH)
print(f'reload check: {check.shape}')

#Modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')

# recreate time columns
df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year
df_clean['hour_of_day'] = df_clean['date'].dt.hour
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)
df_clean['is_rush_hour'] = (
    (df_clean['is_weekend'] == 0) &
    (df_clean['hour_of_day'].between(7, 9) | df_clean['hour_of_day'].between(16, 19))
).astype(int)

# crime severity group
VIOLENT_SEVERE = {'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'KIDNAPPING', 'HUMAN TRAFFICKING'}
VIOLENT_MODERATE = {'ROBBERY', 'BATTERY', 'ASSAULT', 'STALKING'}
PROPERTY = {'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON', 'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'}

def assign_severity(crime_type):
    if crime_type in VIOLENT_SEVERE:
        return 'VIOLENT_SEVERE'
    elif crime_type in VIOLENT_MODERATE:
        return 'VIOLENT_MODERATE'
    elif crime_type in PROPERTY:
        return 'PROPERTY'
    else:
        return 'OTHER'

df_clean['crime_severity_group'] = df_clean['primary_type'].map(assign_severity)

# drop and recompute target encoded features
if 'location_risk_score' in df_clean.columns:
    df_clean = df_clean.drop(columns=['location_risk_score'])
if 'district_arrest_rate' in df_clean.columns:
    df_clean = df_clean.drop(columns=['district_arrest_rate'])

loc_arrest_rate = df_clean.groupby('location_description')['arrest'].mean().rename('location_risk_score')
df_clean = df_clean.join(loc_arrest_rate, on='location_description')

district_arrest_rate = df_clean.groupby('district')['arrest'].mean().rename('district_arrest_rate')
df_clean = df_clean.join(district_arrest_rate, on='district')

print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#Time-based train/val/test split

In [ ]:
# sort by date to prevent leakage
df_clean = df_clean.sort_values('date').reset_index(drop=True)

# 2024 = train, jan-sep 2025 = val, oct 2025 - 2026 = test
train = df_clean[df_clean['year'] == 2024]
val   = df_clean[(df_clean['year'] == 2025) & (df_clean['month'] <= 9)]
test  = df_clean[(df_clean['year'] == 2025) & (df_clean['month'] >= 10) | (df_clean['year'] == 2026)]

print(f'train: {len(train):,} rows')
print(f'val:   {len(val):,} rows')
print(f'test:  {len(test):,} rows')

#Feature Matrix

In [ ]:
# features used for both models
FEATURES = [
    'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'is_rush_hour',
    'location_risk_score', 'district_arrest_rate',
    'location_unknown_flag', 'coords_missing_flag', 'domestic'
]

# encode crime severity as numeric
severity_map = {'VIOLENT_SEVERE': 3, 'VIOLENT_MODERATE': 2, 'PROPERTY': 1, 'OTHER': 0}
for split in [train, val, test]:
    split['severity_encoded'] = split['crime_severity_group'].map(severity_map)

FEATURES = FEATURES + ['severity_encoded']

X_train = train[FEATURES].fillna(0)
X_val   = val[FEATURES].fillna(0)
X_test  = test[FEATURES].fillna(0)

# binary target: arrest
y_arrest_train = train['arrest'].astype(int)
y_arrest_val   = val['arrest'].astype(int)
y_arrest_test  = test['arrest'].astype(int)

# multiclass target: primary_type_grouped
le = LabelEncoder()
y_type_train = le.fit_transform(train['primary_type_grouped'])
y_type_val   = le.transform(val['primary_type_grouped'])
y_type_test  = le.transform(test['primary_type_grouped'])

print('features:', FEATURES)
print(f'X_train shape: {X_train.shape}')

#Task 1: Arrest prediction (Logistic Regression baseline)

In [ ]:
print('training logistic regression for arrest prediction...')
lr_arrest = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_arrest.fit(X_train, y_arrest_train)

y_pred_lr = lr_arrest.predict(X_val)
y_prob_lr = lr_arrest.predict_proba(X_val)[:, 1]

print('Logistic Regression — Arrest Prediction (validation set)')
print(classification_report(y_arrest_val, y_pred_lr, target_names=['No Arrest', 'Arrest']))
print(f'ROC-AUC: {roc_auc_score(y_arrest_val, y_prob_lr):.4f}')

#Task 1: Arrest prediction (Random Forest)

In [ ]:
print('training random forest for arrest prediction...')
rf_arrest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_arrest.fit(X_train, y_arrest_train)

y_pred_rf = rf_arrest.predict(X_val)
y_prob_rf = rf_arrest.predict_proba(X_val)[:, 1]

print('Random Forest — Arrest Prediction (validation set)')
print(classification_report(y_arrest_val, y_pred_rf, target_names=['No Arrest', 'Arrest']))
print(f'ROC-AUC: {roc_auc_score(y_arrest_val, y_prob_rf):.4f}')

#Arrest model comparison + confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_arrest_val, y_pred_lr,
    display_labels=['No Arrest', 'Arrest'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(
    y_arrest_val, y_pred_rf,
    display_labels=['No Arrest', 'Arrest'],
    cmap='Blues', ax=axes[1]
)
axes[1].set_title('Random Forest')

plt.suptitle('Arrest Prediction - Confusion Matrices (Validation Set)')
plt.tight_layout()
plt.savefig('viz_arrest_confusion.png', dpi=150)
plt.show()

# feature importance from random forest
feat_imp = pd.Series(rf_arrest.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
feat_imp.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importance - Arrest Prediction')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('viz_arrest_feature_importance.png', dpi=150)
plt.show()

#Task 2: Crime type prediction (Logistic Regression baseline)

In [ ]:
print('training logistic regression for crime type prediction...')
lr_type = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_type.fit(X_train, y_type_train)

y_pred_lr_type = lr_type.predict(X_val)

print('Logistic Regression — Crime Type Prediction (validation set)')
print(classification_report(y_type_val, y_pred_lr_type, target_names=le.classes_))

#Task 2: Crime type prediction (Random Forest)

In [ ]:
print('training random forest for crime type prediction...')
rf_type = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_type.fit(X_train, y_type_train)

y_pred_rf_type = rf_type.predict(X_val)

print('Random Forest — Crime Type Prediction (validation set)')
print(classification_report(y_type_val, y_pred_rf_type, target_names=le.classes_))

#Crime type confusion matrix

In [ ]:
# logistic regression confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay.from_predictions(
    y_type_val, y_pred_lr_type,
    display_labels=le.classes_,
    cmap='Blues', ax=ax, xticks_rotation=90,
    colorbar=False
)
disp.text_.flatten()
for text in disp.text_.flatten():
    text.set_fontsize(7)
ax.set_title('Crime Type Prediction - Logistic Regression (Validation Set)')
plt.tight_layout()
plt.savefig('viz_type_confusion_lr.png', dpi=150)
plt.show()

# random forest confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay.from_predictions(
    y_type_val, y_pred_rf_type,
    display_labels=le.classes_,
    cmap='Blues', ax=ax, xticks_rotation=90,
    colorbar=False
)
for text in disp.text_.flatten():
    text.set_fontsize(7)
ax.set_title('Crime Type Prediction - Random Forest (Validation Set)')
plt.tight_layout()
plt.savefig('viz_type_confusion_rf.png', dpi=150)
plt.show()

In [ ]:
!pip install -q xgboost shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df_clean = pd.read_parquet('/content/chicago_crimes_clean.parquet')

df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year
df_clean['hour_of_day'] = df_clean['date'].dt.hour
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)
df_clean['is_rush_hour'] = (
    (df_clean['is_weekend'] == 0) &
    (df_clean['hour_of_day'].between(7, 9) | df_clean['hour_of_day'].between(16, 19))
).astype(int)

VIOLENT_SEVERE = {'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'KIDNAPPING', 'HUMAN TRAFFICKING'}
VIOLENT_MODERATE = {'ROBBERY', 'BATTERY', 'ASSAULT', 'STALKING'}
PROPERTY = {'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON', 'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'}

def assign_severity(crime_type):
    if crime_type in VIOLENT_SEVERE:
        return 'VIOLENT_SEVERE'
    elif crime_type in VIOLENT_MODERATE:
        return 'VIOLENT_MODERATE'
    elif crime_type in PROPERTY:
        return 'PROPERTY'
    else:
        return 'OTHER'

df_clean['crime_severity_group'] = df_clean['primary_type'].map(assign_severity)

if 'location_risk_score' in df_clean.columns:
    df_clean = df_clean.drop(columns=['location_risk_score'])
if 'district_arrest_rate' in df_clean.columns:
    df_clean = df_clean.drop(columns=['district_arrest_rate'])

loc_arrest_rate = df_clean.groupby('location_description')['arrest'].mean().rename('location_risk_score')
df_clean = df_clean.join(loc_arrest_rate, on='location_description')

district_arrest_rate = df_clean.groupby('district')['arrest'].mean().rename('district_arrest_rate')
df_clean = df_clean.join(district_arrest_rate, on='district')

print(f'loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

#Train/val/test split

In [ ]:
df_clean = df_clean.sort_values('date').reset_index(drop=True)

train = df_clean[df_clean['year'] == 2024]
val   = df_clean[(df_clean['year'] == 2025) & (df_clean['month'] <= 9)]
test  = df_clean[(df_clean['year'] == 2025) & (df_clean['month'] >= 10) | (df_clean['year'] == 2026)]

print(f'train: {len(train):,}')
print(f'val:   {len(val):,}')
print(f'test:  {len(test):,}')

#Fearure Matrix

In [ ]:
severity_map = {'VIOLENT_SEVERE': 3, 'VIOLENT_MODERATE': 2, 'PROPERTY': 1, 'OTHER': 0}
for split in [train, val, test]:
    split['severity_encoded'] = split['crime_severity_group'].map(severity_map)

FEATURES = [
    'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'is_rush_hour',
    'location_risk_score', 'district_arrest_rate',
    'location_unknown_flag', 'coords_missing_flag',
    'domestic', 'severity_encoded'
]

X_train = train[FEATURES].fillna(0)
X_val   = val[FEATURES].fillna(0)
X_test  = test[FEATURES].fillna(0)

y_arrest_train = train['arrest'].astype(int)
y_arrest_val   = val['arrest'].astype(int)
y_arrest_test  = test['arrest'].astype(int)

le = LabelEncoder()
y_type_train = le.fit_transform(train['primary_type_grouped'])
y_type_val   = le.transform(val['primary_type_grouped'])
y_type_test  = le.transform(test['primary_type_grouped'])

print('features ready')
print(f'X_train: {X_train.shape}')

#XGBoost arrest prediction

In [ ]:
print('training xgboost for arrest prediction...')
xgb_arrest = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_arrest_train == 0).sum() / (y_arrest_train == 1).sum(),
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    verbosity=0
)
xgb_arrest.fit(
    X_train, y_arrest_train,
    eval_set=[(X_val, y_arrest_val)],
    verbose=False
)

y_pred_xgb_arrest = xgb_arrest.predict(X_val)
y_prob_xgb_arrest = xgb_arrest.predict_proba(X_val)[:, 1]

print('XGBoost — Arrest Prediction (validation set)')
print(classification_report(y_arrest_val, y_pred_xgb_arrest, target_names=['No Arrest', 'Arrest']))
print(f'ROC-AUC: {roc_auc_score(y_arrest_val, y_prob_xgb_arrest):.4f}')

# Hyperparameter tuning arrest (random search)

In [ ]:
print('running random search for arrest prediction...')

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=(y_arrest_train == 0).sum() / (y_arrest_train == 1).sum(),
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='auc'
)

search_arrest = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_arrest.fit(X_train, y_arrest_train)

print(f'best params: {search_arrest.best_params_}')
print(f'best CV ROC-AUC: {search_arrest.best_score_:.4f}')

best_xgb_arrest = search_arrest.best_estimator_
y_pred_best_arrest = best_xgb_arrest.predict(X_val)
y_prob_best_arrest = best_xgb_arrest.predict_proba(X_val)[:, 1]

print('\nBest XGBoost — Arrest Prediction (validation set)')
print(classification_report(y_arrest_val, y_pred_best_arrest, target_names=['No Arrest', 'Arrest']))
print(f'ROC-AUC: {roc_auc_score(y_arrest_val, y_prob_best_arrest):.4f}')

#XGBoost crime type prediction + tuning

In [ ]:
print('training xgboost for crime type prediction...')
xgb_type = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_type.fit(X_train, y_type_train)

y_pred_xgb_type = xgb_type.predict(X_val)
print('XGBoost — Crime Type Prediction (validation set)')
print(classification_report(y_type_val, y_pred_xgb_type, target_names=le.classes_))

#Final evaluation on test set

In [ ]:
print('final evaluation on held-out test set')
print('(run this only once — do not tune after seeing these numbers)\n')

# arrest
y_pred_final_arrest = best_xgb_arrest.predict(X_test)
y_prob_final_arrest = best_xgb_arrest.predict_proba(X_test)[:, 1]
print('ARREST PREDICTION - TEST SET')
print(classification_report(y_arrest_test, y_pred_final_arrest, target_names=['No Arrest', 'Arrest']))
print(f'ROC-AUC: {roc_auc_score(y_arrest_test, y_prob_final_arrest):.4f}')

# crime type
y_pred_final_type = xgb_type.predict(X_test)
print('\nCRIME TYPE PREDICTION - TEST SET')
print(classification_report(y_type_test, y_pred_final_type, target_names=le.classes_))

#SHAP feature importance (arrest model)

In [ ]:
print('computing SHAP values (this may take a few minutes)...')

# sample 5000 rows for SHAP — computing on full val set is very slow
shap_sample = X_val.sample(5000, random_state=42)

explainer = shap.TreeExplainer(best_xgb_arrest)
shap_values = explainer.shap_values(shap_sample)

plt.figure()
shap.summary_plot(shap_values, shap_sample, show=False)
plt.title('SHAP Feature Importance - Arrest Prediction')
plt.tight_layout()
plt.savefig('viz_shap_arrest.png', dpi=150, bbox_inches='tight')
plt.show()

#Model comparison summary table

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# reload baseline models from checkpoint if needed, otherwise just summarize
results = pd.DataFrame({
    'model': ['Logistic Regression', 'Random Forest', 'XGBoost (default)', 'XGBoost (tuned)'],
    'task': ['Arrest'] * 4,
    'roc_auc': [
        roc_auc_score(y_arrest_val, lr_arrest.predict_proba(X_val)[:, 1]),
        roc_auc_score(y_arrest_val, rf_arrest.predict_proba(X_val)[:, 1]),
        roc_auc_score(y_arrest_val, y_prob_xgb_arrest),
        roc_auc_score(y_arrest_val, y_prob_best_arrest),
    ],
    'f1_macro': [
        f1_score(y_arrest_val, lr_arrest.predict(X_val), average='macro'),
        f1_score(y_arrest_val, rf_arrest.predict(X_val), average='macro'),
        f1_score(y_arrest_val, y_pred_xgb_arrest, average='macro'),
        f1_score(y_arrest_val, y_pred_best_arrest, average='macro'),
    ]
})
results['roc_auc'] = results['roc_auc'].round(4)
results['f1_macro'] = results['f1_macro'].round(4)
print(results.to_string(index=False))

#save files

In [ ]:

import joblib

# save models
joblib.dump(best_xgb_arrest, 'model_arrest.pkl')
joblib.dump(xgb_type, 'model_type.pkl')
joblib.dump(le, 'label_encoder.pkl')

# save the feature list so streamlit uses exact same columns
joblib.dump(FEATURES, 'features.pkl')

# save location and district risk scores for streamlit to use
loc_risk = df_clean.groupby('location_description')['arrest'].mean().to_dict()
district_risk = df_clean.groupby('district')['arrest'].mean().to_dict()
joblib.dump(loc_risk, 'loc_risk.pkl')
joblib.dump(district_risk, 'district_risk.pkl')

print('all models and artifacts saved')

NameError: name 'best_xgb_arrest' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
files_to_save = [
    'model_arrest.pkl', 'model_type.pkl', 'label_encoder.pkl',
    'features.pkl', 'loc_risk.pkl', 'district_risk.pkl',
    'chicago_crimes_clean.parquet'
]
save_dir = '/content/drive/MyDrive/CS451_FinalProject/'
os.makedirs(save_dir, exist_ok=True)

for f in files_to_save:
    shutil.copy(f'/content/{f}', save_dir + f)
    print(f'saved {f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'os' is not defined

In [ ]:
import shutil, os

files_to_load = [
    'model_arrest.pkl', 'model_type.pkl', 'label_encoder.pkl',
    'features.pkl', 'loc_risk.pkl', 'district_risk.pkl',
    'chicago_crimes_clean.parquet', 'cpd_districts.jpeg'
]
for f in files_to_load:
    shutil.copy(f'/content/drive/MyDrive/CS451_FinalProject/{f}', f'/content/{f}')
    print(f'loaded {f}')

loaded model_arrest.pkl
loaded model_type.pkl
loaded label_encoder.pkl
loaded features.pkl
loaded loc_risk.pkl
loaded district_risk.pkl
loaded chicago_crimes_clean.parquet
loaded cpd_districts.jpeg


#Streamlit

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib

model_arrest = joblib.load('model_arrest.pkl')
model_type = joblib.load('model_type.pkl')
le = joblib.load('label_encoder.pkl')
features = joblib.load('features.pkl')
loc_risk = joblib.load('loc_risk.pkl')
district_risk = joblib.load('district_risk.pkl')
df = pd.read_parquet('chicago_crimes_clean.parquet')

community_area_names = {
    1: 'Rogers Park', 2: 'West Ridge', 3: 'Uptown', 4: 'Lincoln Square',
    5: 'North Center', 6: 'Lake View', 7: 'Lincoln Park', 8: 'Near North Side',
    9: 'Edison Park', 10: 'Norwood Park', 11: 'Jefferson Park', 12: 'Forest Glen',
    13: 'North Park', 14: 'Albany Park', 15: 'Portage Park', 16: 'Irving Park',
    17: 'Dunning', 18: 'Montclare', 19: 'Belmont Cragin', 20: 'Hermosa',
    21: 'Avondale', 22: 'Logan Square', 23: 'Humboldt Park', 24: 'West Town',
    25: 'Austin', 26: 'West Garfield Park', 27: 'East Garfield Park', 28: 'Near West Side',
    29: 'North Lawndale', 30: 'South Lawndale', 31: 'Lower West Side', 32: 'Loop',
    33: 'Near South Side', 34: 'Armour Square', 35: 'Douglas', 36: 'Oakland',
    37: 'Fuller Park', 38: 'Grand Boulevard', 39: 'Kenwood', 40: 'Washington Park',
    41: 'Hyde Park', 42: 'Woodlawn', 43: 'South Shore', 44: 'Chatham',
    45: 'Avalon Park', 46: 'South Chicago', 47: 'Burnside', 48: 'Calumet Heights',
    49: 'Roseland', 50: 'Pullman', 51: 'South Deering', 52: 'East Side',
    53: 'West Pullman', 54: 'Riverdale', 55: 'Hegewisch', 56: 'Garfield Ridge',
    57: 'Archer Heights', 58: 'Brighton Park', 59: 'McKinley Park', 60: 'Bridgeport',
    61: 'New City', 62: 'West Elsdon', 63: 'Gage Park', 64: 'Clearing',
    65: 'West Lawn', 66: 'Chicago Lawn', 67: 'West Englewood', 68: 'Englewood',
    69: 'Greater Grand Crossing', 70: 'Ashburn', 71: 'Auburn Gresham',
    72: 'Beverly', 73: 'Washington Heights', 74: 'Mount Greenwood',
    75: 'Morgan Park', 76: 'OHare', 77: 'Edgewater'
}
name_to_area = {v: k for k, v in community_area_names.items()}

VIOLENT_SEVERE = {'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'KIDNAPPING', 'HUMAN TRAFFICKING'}
VIOLENT_MODERATE = {'ROBBERY', 'BATTERY', 'ASSAULT', 'STALKING'}
PROPERTY = {'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON', 'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'}

def assign_severity(crime_type):
    if crime_type in VIOLENT_SEVERE:
        return 3
    elif crime_type in VIOLENT_MODERATE:
        return 2
    elif crime_type in PROPERTY:
        return 1
    else:
        return 0

st.set_page_config(page_title='Chicago Crime Explorer', layout='wide')
st.title('Chicago Crime Explorer')
st.markdown('Predict arrest likelihood and explore crime patterns by neighborhood.')

tab1, tab2 = st.tabs(['Predict', 'Neighborhood Explorer'])

with tab1:
    st.header('Arrest Likelihood Prediction')
    st.markdown('Enter details about an incident to predict whether an arrest will be made.')

    col1, col2, col3 = st.columns(3)

    with col1:
        hour = st.slider('Hour of Day', 0, 23, 12)
        day = st.selectbox('Day of Week', ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
        month = st.selectbox('Month', ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])

    with col2:
        location = st.selectbox('Location Type', sorted(loc_risk.keys()))
        with st.expander('View CPD District Map'):
            st.image('cpd_districts.jpeg', caption='Chicago Police Department Districts', width=600)
        district_names = {
            1.0: '1 - Central', 2.0: '2 - Wentworth', 3.0: '3 - Grand Crossing',
            4.0: '4 - South Chicago', 5.0: '5 - Calumet', 6.0: '6 - Gresham',
            7.0: '7 - Englewood', 8.0: '8 - Chicago Lawn', 9.0: '9 - Deering',
            10.0: '10 - Ogden', 11.0: '11 - Harrison', 12.0: '12 - Near West',
            14.0: '14 - Shakespeare', 15.0: '15 - Austin', 16.0: '16 - Jefferson Park',
            17.0: '17 - Albany Park', 18.0: '18 - Near North', 19.0: '19 - Town Hall',
            20.0: '20 - Lincoln', 22.0: '22 - Morgan Park', 24.0: '24 - Rogers Park',
            25.0: '25 - Grand Central'
        }
        valid_districts = sorted([d for d in district_risk.keys() if d != -1])
        district_labels = [district_names.get(d, str(d)) for d in valid_districts]
        district_label = st.selectbox('District', district_labels)
        district = valid_districts[district_labels.index(district_label)]
        domestic = st.checkbox('Domestic Incident')

    with col3:
        crime_type = st.selectbox('Crime Type', sorted(df['primary_type_grouped'].unique()))

    if st.button('Predict', type='primary'):
        day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
        month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

        day_num = day_map[day]
        month_num = month_map[month]
        is_weekend = 1 if day_num >= 5 else 0
        is_rush = 1 if (not is_weekend and (7 <= hour <= 9 or 16 <= hour <= 19)) else 0
        loc_score = loc_risk.get(location, 0.5)
        dist_score = district_risk.get(district, 0.5)
        severity = assign_severity(crime_type)

        input_data = pd.DataFrame([{
            'hour_of_day': hour,
            'day_of_week': day_num,
            'is_weekend': is_weekend,
            'month': month_num,
            'is_rush_hour': is_rush,
            'location_risk_score': loc_score,
            'district_arrest_rate': dist_score,
            'location_unknown_flag': 0,
            'coords_missing_flag': 0,
            'domestic': int(domestic),
            'severity_encoded': severity
        }])[features]

        arrest_prob = model_arrest.predict_proba(input_data)[0][1]

        st.subheader('Results')
        st.metric('Arrest Probability', f'{arrest_prob*100:.1f}%')
        if arrest_prob > 0.5:
            st.success('Arrest likely')
        else:
            st.warning('Arrest unlikely')

with tab2:
    st.header('Neighborhood Crime Explorer')

    selected_hood = st.selectbox('Select Neighborhood', sorted(community_area_names.values()))
    area_num = name_to_area[selected_hood]

    hood_data = df[df['community_area'] == area_num]

    if len(hood_data) == 0:
        st.warning('No data for this neighborhood.')
    else:
        col1, col2, col3 = st.columns(3)
        col1.metric('Total Incidents', f"{len(hood_data):,}")
        col2.metric('Arrest Rate', f"{hood_data['arrest'].mean()*100:.1f}%")
        col3.metric('City Avg Arrest Rate', f"{df['arrest'].mean()*100:.1f}%")

        st.subheader('Crime Type Breakdown')
        crime_counts = hood_data['primary_type'].value_counts().head(10)
        st.bar_chart(crime_counts)

        st.subheader('Top Crime Types')
        crime_table = pd.DataFrame({
            'Crime Type': crime_counts.index,
            'Count': crime_counts.values,
            'Pct of Neighborhood': (crime_counts.values / len(hood_data) * 100).round(1)
        })
        st.dataframe(crime_table, use_container_width=True)

        st.subheader('Arrest Rate by Crime Type')
        arrest_by_type = (
            hood_data.groupby('primary_type')['arrest']
            .agg(['mean', 'count'])
            .rename(columns={'mean': 'Arrest Rate', 'count': 'Count'})
            .query('Count >= 10')
            .sort_values('Arrest Rate', ascending=False)
        )
        arrest_by_type['Arrest Rate'] = (arrest_by_type['Arrest Rate'] * 100).round(1)
        st.dataframe(arrest_by_type, use_container_width=True)
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print('app.py updated')

app.py updated


In [ ]:
!pip install -q streamlit pyngrok

from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token("3CcqJryAdhIghsXLTcTT9oWNvsR_4V1HhkgDLyYxDDr1sEzxQ")

proc = subprocess.Popen(['streamlit', 'run', '/content/app.py',
                         '--server.port', '8501',
                         '--server.headless', 'true'])
time.sleep(5)
tunnel = ngrok.connect(8501)
print('Streamlit URL:', tunnel.public_url)

Streamlit URL: https://conical-device-spirits.ngrok-free.dev


In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/CS451_FinalProject/cpd_districts.jpeg', '/content/cpd_districts.jpeg')
print('done')

done


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/CS451_FinalProject/'))

In [ ]:
from google.colab import files
files.download('viz_shap_arrest.png')

FileNotFoundError: Cannot find file: viz_shap_arrest.png